<a href="https://colab.research.google.com/github/SmritiD2004/RAG--IBMSkillsbuild/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 33.5 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np
import google.generativeai as genai

from sentence_transformers import SentenceTransformer
from google.colab import files
from google.colab import userdata

In [ ]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini initialized successfully!")

Gemini initialized successfully!


In [ ]:
user_query = input("Ask a question: ")

response = model.generate_content(user_query)

initial_response = response.text

print("\nBaseline Response (Without RAG):\n")
print(initial_response)

Ask a question: Hi. I'm a citizen of Romania. Do I need a Schengen visa if I travel to France?

Baseline Response (Without RAG):

No, as a citizen of Romania, you **do not need a Schengen visa** to travel to France.

Here's why:

1.  **EU Citizenship:** Romania is a member of the European Union. As an EU citizen, you have the right to free movement within the EU, which includes France.
2.  **Schengen Area:** While Romania only recently joined the Schengen Area for air and sea borders (as of March 31, 2024), this simply means you'll no longer go through border checks when flying or taking a ferry between Romania and other Schengen countries. Your right to visa-free travel to France already existed due to your EU citizenship.

You only need your valid Romanian passport or national ID card to travel to France for short stays (tourism, visiting, etc.).


In [ ]:
uploaded = files.upload()

doc_path = list(uploaded.keys())[0]

with open(doc_path, "r", encoding="utf-8") as f:
    document_text = f.read()

print("Document Loaded Successfully!")

Saving External_KB_for_RAG (1).txt to External_KB_for_RAG (1).txt
Document Loaded Successfully!


In [ ]:
def split_into_chunks(text, chunk_size=300, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = split_into_chunks(document_text)

print(f"Document split into {len(chunks)} chunks.")

Document split into 10 chunks.


In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(chunks)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(10, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

stored_chunks = chunks

print(f"Stored {len(stored_chunks)} chunks in FAISS.")

Stored 10 chunks in FAISS.


In [ ]:
query_embedding = embed_model.encode([user_query])

k = 3

distances, indices = index.search(np.array(query_embedding), k)

retrieved_context = "\n\n".join(
    [stored_chunks[i] for i in indices[0]]
)

print("Retrieved Context:\n")
print(retrieved_context)

Retrieved Context:

, which will be processed within three days of the application. 

 

User query (posed on November 30, 2024): 

Hi. I’m a citizen of Romania. Do I need a Schengen visa if I travel to France? 

System response (pre-RAG implementation): 

Yes, you need to apply for a Schengen visa to enter France. 

S

 to apply for a Schengen visa to enter France. 

System response (post-RAG implementation): 

If you are traveling to France after January 1, 2025, you will no longer need to apply for a Schengen visa.  

 

 ETIAS permit before traveling. This permit is required for short-term stays (up to 90 days in a 180-day period) and is valid for up to three years or until the traveler’s passport expires. It costs €7 for applications aged 18 to 70. 

 

Romania and Bulgaria joining Schengen 

As of January 1, 2025


In [ ]:
prompt = f"""
You are a helpful travel assistant.

Answer ONLY using the information provided in the context below.

If the answer is not present in the context, say:
"I don't have enough information in the provided documents."

Context:
{retrieved_context}

Question:
{user_query}
"""

response = model.generate_content(prompt)

rag_response = response.text

print("\nRAG Response:\n")
print(rag_response)


RAG Response:

If you are traveling to France after January 1, 2025, you will no longer need to apply for a Schengen visa. However, you will need an ETIAS permit before traveling, which is required for short-term stays (up to 90 days in a 180-day period) and is valid for up to three years or until your passport expires. It costs €7 for applications aged 18 to 70.

I don't have enough information in the provided documents to determine if a Schengen visa is needed for travel to France *before* January 1, 2025.


In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"""
# Original Query

**{user_query}**

---

# Without RAG

{initial_response}

---

# With RAG

{rag_response}
"""))


# Original Query

**Hi. I'm a citizen of Romania. Do I need a Schengen visa if I travel to France?**

---

# Without RAG

No, as a citizen of Romania, you **do not need a Schengen visa** to travel to France.

Here's why:

1.  **EU Citizenship:** Romania is a member of the European Union. As an EU citizen, you have the right to free movement within the EU, which includes France.
2.  **Schengen Area:** While Romania only recently joined the Schengen Area for air and sea borders (as of March 31, 2024), this simply means you'll no longer go through border checks when flying or taking a ferry between Romania and other Schengen countries. Your right to visa-free travel to France already existed due to your EU citizenship.

You only need your valid Romanian passport or national ID card to travel to France for short stays (tourism, visiting, etc.).

---

# With RAG

If you are traveling to France after January 1, 2025, you will no longer need to apply for a Schengen visa. However, you will need an ETIAS permit before traveling, which is required for short-term stays (up to 90 days in a 180-day period) and is valid for up to three years or until your passport expires. It costs €7 for applications aged 18 to 70.

I don't have enough information in the provided documents to determine if a Schengen visa is needed for travel to France *before* January 1, 2025.
